# KAVI audience enrichment

Answers the finder's question *"Who is it for?"* for every resource. Run the cells from top to bottom.

The notebook reads `data/cleaned/resources_extended.csv` (from `scripts/pdf/extend_clean_resources.py`) and writes:

- `data/cleaned/resource_audiences.csv` - one row per `resource_id`, to be joined onto the resource table
- `data/review/audience_review.csv` - a random sample for a manual check of the rules

It adds three things to the PDF-based `pdf_audiences`:

1. **`description_audiences`** - audiences named in the description, including two the PDF has no chapter for (*Survivors of violence*, *People without homes*).
2. **`audience_restriction`** - whether the resource is limited to certain people (`restricted`), says it is open to everyone (`open`), says both (`mixed`, needs review), or says neither (`not_stated`).
3. **`audiences_all`** - `pdf_audiences` and `description_audiences` together, for the finder's filter.

A blank `pdf_audiences` only means the organization is not listed in one of the PDF's audience chapters. It does not mean the resource is open to everyone. That is why `audience_restriction` is a separate column.

In [ ]:
import os
import re

import pandas as pd

IN_FILE = "../../data/cleaned/resources_extended.csv"
OUT_FILE = "../../data/cleaned/resource_audiences.csv"
REVIEW_FILE = "../../data/review/audience_review.csv"
os.makedirs(os.path.dirname(REVIEW_FILE), exist_ok=True)

df = pd.read_csv(IN_FILE, dtype=str, keep_default_na=False)
print(len(df), "rows,", (df["pdf_audiences"] != "").sum(), "with pdf_audiences")

## 1. Audience patterns

One regular expression per audience, matched case-insensitively against the description. The names of the first nine are the PDF's audience chapters, so they line up with `pdf_audiences`.

Patterns match *mentions*. Step 2 decides whether a mention is a restriction.

In [ ]:
AUDIENCES = {
    "Youth": r"\byouth|young (people|adults)|\bteens?\b|adolescent|ages? 1[0-9]\b",
    "Women": r"\bwomen\b|\bgirls?\b",
    "LGBTQI+": r"lgbt|transgender|\btrans\b|queer|gender[- ](non-?conforming|fluid|expansive)",
    "Veterans": r"veteran",
    "Immigrants": r"immigra|undocumented|asylum|refugee",
    "Older People": r"\bseniors?\b|older (adults|people|new yorkers)|elderly|aged? 6[05]",
    "People with Disabilities": r"disabilit|\bdeaf\b|\bblind\b|hard of hearing|visually impaired",
    "Parents & Caregivers": r"\bparents?\b|caregiver|\bfamil(y|ies)\b",
    "Formerly Incarcerated People": r"formerly incarcerated|justice[- ]involved|re-?entry|incarcerat|parole|probation",
    "Survivors of violence": r"domestic violence|intimate partner|dating (abuse|violence)|sexual (assault|abuse|violence)|survivors?\b|trafficking",
    "People without homes": r"homeless|unhoused|without homes|street homeless",
}

def found_audiences(text):
    return [name for name, pattern in AUDIENCES.items() if re.search(pattern, text, re.I)]

df["description_audiences"] = df["description"].apply(lambda d: "; ".join(found_audiences(d)))

## 2. Restricted, open, or not stated

Heuristics, meant to be improved after the manual check:

- **restricted**: an audience is named in the *first sentence* (that is usually who the resource is for), or the text gives an age range, or says *only* / *must be* / *eligible*.
- **open**: the text says it serves anyone (*anyone*, *all New Yorkers*, *regardless of* …).
- **mixed**: both apply, e.g. "open to all, with a program for youth". Flagged for review.
- **not_stated**: neither.

A mention later in a list ("services include … services for homeless youth") is kept in `description_audiences` but does not by itself make the resource `restricted`.

In [ ]:
OPEN = r"\b(anyone|everyone|all new yorkers|all (ages|people|residents)|regardless of|open to (all|the public)|no (eligibility|referral) (requirements?|needed))\b"
LIMIT = r"\b(only|exclusively|must be|eligible|eligibility)\b|ages? \d{1,2}\s*(-|–|to)\s*\d{1,2}"

def first_sentence(text):
    return re.split(r"(?<=[.!?])\s", text.strip(), maxsplit=1)[0]

def restriction(row):
    text = row["description"]
    named_first = bool(found_audiences(first_sentence(text)))
    limited = named_first or bool(re.search(LIMIT, text, re.I))
    is_open = bool(re.search(OPEN, text, re.I))
    if limited and is_open:
        return "mixed"
    if limited:
        return "restricted"
    if is_open:
        return "open"
    return "not_stated"

df["audience_restriction"] = df.apply(restriction, axis=1)
df["audience_restriction"].value_counts()

## 3. Combine with the PDF audiences

In [ ]:
def split(value):
    return [v.strip() for v in value.split(";") if v.strip()]

def combine(row):
    pdf, desc = split(row["pdf_audiences"]), split(row["description_audiences"])
    return "; ".join(dict.fromkeys(pdf + desc))

def source(row):
    has_pdf, has_desc = bool(row["pdf_audiences"]), bool(row["description_audiences"])
    return {(True, True): "pdf+description", (True, False): "pdf", (False, True): "description"}.get((has_pdf, has_desc), "")

df["audiences_all"] = df.apply(combine, axis=1)
df["audience_source"] = df.apply(source, axis=1)
df["audience_needs_review"] = (df["audience_restriction"] == "mixed") | (df["audience_source"] == "description")

print("rows with any audience:", (df["audiences_all"] != "").sum(), "of", len(df))
df["audience_source"].replace("", "(none)").value_counts()

## 4. Cross-check against the descriptions

Where does the description disagree with the PDF? These comparisons are the quickest way to find weak patterns.

In [ ]:
pdf_only = df[(df["pdf_audiences"] != "") & (df["description_audiences"] == "")]
desc_only = df[(df["pdf_audiences"] == "") & (df["description_audiences"] != "")]
print(len(pdf_only), "rows: PDF names an audience, description does not")
print(len(desc_only), "rows: description names an audience, PDF does not")

pd.set_option("display.max_colwidth", 200)
desc_only[["resource_id", "org_name", "description_audiences", "audience_restriction", "description"]].sample(10, random_state=1)

## 5. Save, plus a review sample

`audience_review.csv` is a random sample of 50 rows with an audience or a restriction, with empty columns for a person's verdict. Check it, then sharpen the patterns above and run again.

In [ ]:
cols = ["resource_id", "org_name", "pdf_audiences", "description_audiences", "audiences_all",
        "audience_restriction", "audience_source", "audience_needs_review"]
df[cols].to_csv(OUT_FILE, index=False)

sample = df[(df["audiences_all"] != "") | (df["audience_restriction"] != "not_stated")].sample(50, random_state=42)
review = sample[["resource_id", "org_name", "description"] + cols[2:6]].copy()
review["human_verdict"] = ""        # agree / change / unsure
review["human_audiences"] = ""      # only for change: the correct list, separated by "; "
review["human_restriction"] = ""    # only for change: restricted / open / not_stated
review["human_notes"] = ""
review.sort_values("resource_id", key=lambda s: s.astype(int)).to_csv(REVIEW_FILE, index=False)

print(len(df), "rows written to", OUT_FILE)
print(len(review), "rows written to", REVIEW_FILE)